In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sys.version_info(major=3, minor=10, micro=12, releaselevel='final', serial=0)
matplotlib 3.10.8
numpy 1.26.4
pandas 2.2.3
sklearn 1.7.2
torch 2.8.0+cu128
cuda:0


## 数据准备

In [2]:
# !wget https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt

In [3]:
# https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt
#文件已经下载好了
with open("./shakespeare.txt", "r", encoding="utf8") as file:
    text = file.read()

print("length", len(text))
print(text[0:100])

length 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


### 构造字典

In [4]:
# 1. generate vocab
# 2. build mapping char->id
# 3. data -> id_data  把数据都转为id
# 4. a b c d [EOS] -> [BOS] b c d  预测下一个字符生成的模型，也就是输入是a，输出就是b

#去重，留下独立字符，并排序（排序是为了好看）
vocab = sorted(set(text))
print(len(vocab))
print(vocab)

65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [5]:
for idx,char in enumerate(['h','o','w']):
    print(idx,char)

0 h
1 o
2 w


In [6]:
#每个字符都编好号，enumerate对每一个位置编号，生成的是列表中是元组，下面字典生成式
char2idx = {char:idx for idx, char in enumerate(vocab)}
print(char2idx)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [7]:
# 把vocab从列表变为ndarray ，用下标表示id
idx2char = np.array(vocab)
print(idx2char)

['\n' ' ' '!' '$' '&' "'" ',' '-' '.' '3' ':' ';' '?' 'A' 'B' 'C' 'D' 'E'
 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S' 'T' 'U' 'V' 'W'
 'X' 'Y' 'Z' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i' 'j' 'k' 'l' 'm' 'n' 'o'
 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z']


In [8]:
#把字符都转换为id
text_as_int = np.array([char2idx[c] for c in text])
print(text_as_int.shape)
print(len(text_as_int))
print(text_as_int[0:10])
print(text[0:10])

(1115394,)
1115394
[18 47 56 57 58  1 15 47 58 47]
First Citi


In [9]:
1115394//101

11043

### 把莎士比亚文集分成一个一个的样本

In [10]:
# ### 把莎士比亚文集分成一个一个的样本

# 给 RNN / LSTM 训练 “下一个字符预测” 任务
# 输入:  今天天气
# 输出:  天天气好
# 模型学的是：

# 给定前面字符，预测下一个字符。

# 二、先理解 text_as_int 是什么？
# 假设原始文本是：

# hello world

# 我们先做字符编码：

# {'h':0, 'e':1, 'l':2, 'o':3, ' ':4, 'w':5, 'r':6, 'd':7}

# 变成：

# text_as_int = [0,1,2,2,3,4,5,3,6,2,7]

# 这就是传入 CharDataset 的东西。

# 三、Dataset 部分详细拆解：
# 1️⃣ 初始化
# self.sub_len = seq_length + 1

# 如果：

# seq_length = 100

# 那么：

# sub_len = 101

# 为什么是 101？

# 因为：

# 前 100 个作为输入

# 后 100 个作为输出

# 需要多 1 个字符

# 2️⃣ 样本数量
# self.num_seq = len(text_as_int) // self.sub_len

# 假设：

# 文本长度 = 10000
# sub_len = 101

# 那么：

# num_seq = 10000 // 101 ≈ 99

# 说明：

# 能切出 99 个样本
# 每个样本 101 个字符



# 四、关键部分：collate_fn

# 这一部分是核心。

# DataLoader 默认只会把 batch 拼起来。

# 但你需要：

# 输入：前100字符

# 输出：后100字符

# 所以你自定义了：

# def collat_fct(batch):
# batch 长什么样？

# 假设：

# batch_size = 2

# 那么 batch 是：

# [
#  [c1,c2,c3,...,c101],
#  [c102,c103,...,c202]
# ]

# 每个元素长度 = 101

# 1️⃣ 构造输入
# src_list.append(part[:-1])

# 去掉最后一个字符。

# 长度变 100。

# 2️⃣ 构造输出
# trg_list.append(part[1:])

# 去掉第一个字符。

# 长度也是 100。

# 五、举一个超清晰例子

# 假设样本：

# [10, 20, 30, 40]

# 长度 4（假设 seq_length=3）

# 那么：

# 输入：

# [10, 20, 30]

# 输出：

# [20, 30, 40]

# 训练目标就是：

# 给10预测20
# 给20预测30
# 给30预测40

# 这就是语言模型。


# 九、这个 Dataset 是什么任务？

# 这是：

# Character-level Language Modeling

# 字符级语言模型。

# 和 GPT 本质任务一样，只不过是字符级别。

# 例如：

# OpenAI 的 GPT 是 subword 级别

# 你这个是 character 级别


# 十、总结整个流程

# 文本
# ↓
# 字符转id
# ↓
# 切成 101 长度片段
# ↓
# 前100做输入
# ↓
# 后100做标签
# ↓
# RNN 预测下一个字符

In [11]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    #text_as_int是字符的id列表，seq_length是每个样本的长度
    def __init__(self, text_as_int, seq_length):
        self.sub_len = seq_length + 1 #一个样本的长度
        self.text_as_int = text_as_int
        self.num_seq = len(text_as_int) // self.sub_len #样本的个数
        
    def __getitem__(self, index):#index是样本的索引，返回的是一个样本，比如第一个，就是0-100的字符,总计101个字符
        return self.text_as_int[index * self.sub_len: (index + 1) * self.sub_len]
    
    def __len__(self): #返回样本的个数
        return self.num_seq

#batch是一个列表，列表中的每一个元素是一个样本，有101个字符，前100个是输入，后100个是输出
def collat_fct(batch):
    src_list = [] #输入
    trg_list = [] #输出
    for part in batch:
        src_list.append(part[:-1]) #输入
        trg_list.append(part[1:]) #输出
        
    src_list = np.array(src_list) #把列表转换为ndarray
    trg_list = np.array(trg_list) #把列表转换为ndarray
    return torch.Tensor(src_list).to(dtype=torch.int64), torch.Tensor(trg_list).to(dtype=torch.int64) #返回的是一个元组，元组中的每一个元素是一个torch.Tensor
        
#每个样本的长度是101，也就是100个字符+1个结束符
train_ds = CharDataset(text_as_int, 100)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collat_fct)

In [12]:
for datas, labels in train_dl:
    print(datas.shape)
    print(labels.shape)
    break

torch.Size([64, 100])
torch.Size([64, 100])


## 定义模型

In [13]:
# ## 定义模型
# Character-level Language Model
# 输入一串字符 → 预测每个位置的下一个字符


# 字符ID [64,100]
#       ↓
# Embedding
#       ↓
# [64,100,256]
#       ↓
# RNN
#       ↓
# [64,100,1024]
#       ↓
# Linear
#       ↓
# [64,100,65]
#       ↓
# CrossEntropyLoss


#  整个流程的语义解释
# 字符ID [64,100]
# = 64条文本，每条100个字符

# ↓

# Embedding [64,100,256]
# = 每个字符 → 256维语义向量

# ↓

# RNN [64,100,1024]
# = 每个时间步输出一个1024维上下文表示

# ↓

# Linear [64,100,65]
# = 每个时间步预测65种字符的概率

# ↓

# CrossEntropyLoss
# = 对6400个字符做分类损失


# ① 字符ID → [64,100]
# [batch_size, seq_len]
# | 维度  | 数字         | 含义            |
# | --- | ---------- | ------------- |
# | 64  | batch_size | 一次输入64条文本     |
# | 100 | seq_len    | 每条文本长度为100个字符 |

# ② Embedding → [64,100,256]
# self.embedding = nn.Embedding(vocab_size, embedding_dim)
# [batch_size, seq_len, embedding_dim]
# | 维度  | 数字            | 含义             |
# | --- | ------------- | -------------- |
# | 64  | batch_size    | 64个样本          |
# | 100 | seq_len       | 每个样本100个字符     |
# | 256 | embedding_dim | 每个字符被映射成256维向量 |

#  解释：

# 原来每个字符是：

# 一个数字ID

# 现在变成：

# 一个256维向量

# 也就是说：

# 1个字符 → 256个特征

# ③ RNN → [64,100,1024]
# self.rnn = nn.RNN(embedding_dim, hidden_dim)
# [batch_size, seq_len, hidden_dim]
# | 维度   | 数字         | 含义               |
# | ---- | ---------- | ---------------- |
# | 64   | batch_size | 64条文本            |
# | 100  | seq_len    | 100个时间步          |
# | 1024 | hidden_dim | 每个时间步输出1024维隐藏状态 |

#  重要理解：

# RNN 会在 每个时间步 产生一个 hidden state。

# 所以：

# 每个字符 → 输出一个1024维向量

# 可以理解为：

# RNN 把 256维embedding → 编码成 1024维上下文表示

# ④ Linear → [64,100,65]
# self.fc = nn.Linear(hidden_dim, vocab_size)
# [batch_size, seq_len, vocab_size]
# | 维度  | 数字         | 含义      |
# | --- | ---------- | ------- |
# | 64  | batch_size | 64条样本   |
# | 100 | seq_len    | 100个时间步 |
# | 65  | vocab_size | 预测65种字符 |
#  解释：

# 每个时间步：

# 1024维 → 变成 65维

# 这 65 维表示：

# 对下一个字符属于 65 个字符中哪一个的预测分数（logits）


# ⑤ CrossEntropyLoss

# 输入：

# [64,100,65]  # 预测

# 目标：

# [64,100]     # 正确字符ID

# 内部计算时会：

# x = x.view(-1, vocab_size)     # [6400, 65]
# target = target.view(-1)       # [6400]

# 因为：

# 64 × 100 = 6400

# 也就是：

# 总共预测了6400个字符

In [14]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=1024):
        super(CharRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        #batch_first=True,输入的数据格式是(batch_size, seq_len, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        x = self.embedding(x) #(batch_size, seq_len) -> (batch_size, seq_len, embedding_dim) (64, 100, 256)
        #这里和02的差异是没有只拿最后一个输出，而是把所有的输出都拿出来了
        #(batch_size, seq_len, embedding_dim)->(batch_size, seq_len, hidden_dim)(64, 100, 1024)
        output, hidden = self.rnn(x, hidden)
        x = self.fc(output) #[bs, seq_len, hidden_dim]--->[bs, seq_len, vocab_size] (64, 100,65)
        return x, hidden #x的shape是(batch_size, seq_len, vocab_size)
    
    
vocab_size = len(vocab)

print("{:=^80}".format(" 一层单向 RNN "))       
for key, value in CharRNN(vocab_size).named_parameters():
    print(f"{key:^40}paramerters num: {np.prod(value.shape)}")

=================================== 一层单向 RNN ===================================
            embedding.weight            paramerters num: 16640
            rnn.weight_ih_l0            paramerters num: 262144
            rnn.weight_hh_l0            paramerters num: 1048576
             rnn.bias_ih_l0             paramerters num: 1024
             rnn.bias_hh_l0             paramerters num: 1024
               fc.weight                paramerters num: 66560
                fc.bias                 paramerters num: 65


In [15]:
print(65*256)
print(256*1024)
1024*1024

16640
262144


1048576

In [16]:
sample_inputs = torch.randint(0, vocab_size, (3, 100))
print(sample_inputs.shape)
model = CharRNN(vocab_size)
output=model(sample_inputs)
output[0].shape

torch.Size([3, 100])


torch.Size([3, 100, 65])

In [17]:
output[0].reshape(-1, vocab_size).shape

torch.Size([300, 65])

In [18]:

1024*65

66560

## 训练

In [19]:
class SaveCheckpointsCallback:
    def __init__(self, save_dir, save_step=5000, save_best_only=True):
        """
        Save checkpoints each save_epoch epoch. 
        We save checkpoint by epoch in this implementation.
        Usually, training scripts with pytorch evaluating model and save checkpoint by step.

        Args:
            save_dir (str): dir to save checkpoint
            save_epoch (int, optional): the frequency to save checkpoint. Defaults to 1.
            save_best_only (bool, optional): If True, only save the best model or save each model at every epoch.
        """
        self.save_dir = save_dir
        self.save_step = save_step
        self.save_best_only = save_best_only
        self.best_metrics = -1
        
        # mkdir
        if not os.path.exists(self.save_dir):
            os.mkdir(self.save_dir)
        
    def __call__(self, step, state_dict, metric=None):
        if step % self.save_step > 0:
            return
        
        if self.save_best_only:
            assert metric is not None
            if metric >= self.best_metrics:
                # save checkpoints
                torch.save(state_dict, os.path.join(self.save_dir, "best.ckpt"))
                # update best metrics
                self.best_metrics = metric
        else:
            torch.save(state_dict, os.path.join(self.save_dir, f"{step}.ckpt"))



In [20]:
# 训练
def training(
    model, 
    train_loader, 
    epoch, 
    loss_fct, 
    optimizer, 
    save_ckpt_callback=None,
    stateful=False      # 想用stateful，batch里的数据就必须连续，不能打乱
    ):
    record_dict = {
        "train": [],
    }
    
    global_step = 0
    model.train()
    hidden = None
    with tqdm(total=epoch * len(train_loader)) as pbar:
        for epoch_id in range(epoch):
            # training
            for datas, labels in train_loader:
                datas = datas.to(device)
                labels = labels.to(device)
                # 梯度清空
                optimizer.zero_grad()
                # 模型前向计算,如果数据集打乱了，stateful=False，hidden就要清空
                # 如果数据集没有打乱，stateful=True，hidden就不需要清空
                logits, hidden = model(datas, hidden=hidden if stateful else None)
                # 计算损失,交叉熵损失第一个参数要是二阶张量，第二个参数要是一阶张量，所以要reshape
                loss = loss_fct(logits.reshape(-1, vocab_size), labels.reshape(-1))
                # 梯度回传
                loss.backward()
                # 调整优化器，包括学习率的变动等
                optimizer.step()
 
                loss = loss.cpu().item()
                # record
                
                record_dict["train"].append({
                    "loss": loss, "step": global_step
                })
   
                # 保存模型权重 save model checkpoint
                if save_ckpt_callback is not None:
                    save_ckpt_callback(global_step, model.state_dict(), metric=-loss)
                # udate step
                global_step += 1
                pbar.update(1)
                pbar.set_postfix({"epoch": epoch_id})
        
    return record_dict
        

epoch = 100

model = CharRNN(vocab_size=vocab_size)

# 1. 定义损失函数 采用交叉熵损失 
loss_fct = nn.CrossEntropyLoss()
# 2. 定义优化器 采用 adam
# Optimizers specified in the torch.optim package
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# save best
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")
save_ckpt_callback = SaveCheckpointsCallback("checkpoints/text_generation", save_step=1000, save_best_only=True)


model = model.to(device)


In [ ]:
record = training(
    model,
    train_dl,
    epoch,
    loss_fct,
    optimizer,
    save_ckpt_callback=save_ckpt_callback,
    )

 39%|███▉      | 6734/17300 [00:44<01:10, 150.02it/s, epoch=38]

In [ ]:
plt.plot([i["step"] for i in record["train"][::50]], [i["loss"] for i in record["train"][::50]], label="train")
plt.grid()
plt.show()

## 推理

In [ ]:
#下面的例子是为了说明temperature
my_tensor = torch.tensor([0.4,0.6]) #这里是logits

probs1 = F.softmax(my_tensor, dim=-1)
print(probs1)

In [ ]:
my_tensor = torch.tensor([0.2,0.3])  #现在 temperature是2

probs1 = F.softmax(my_tensor, dim=-1)
print(probs1)

In [ ]:
import torch

# 创建一个概率分布，表示每个类别被选中的概率
# 这里我们有一个简单的四个类别的概率分布
prob_dist = torch.tensor([0.1, 0.45, 0.35, 0.1])

# 使用 multinomial 进行抽样
# num_samples 表示要抽取的样本数量
num_samples = 5

# 抽取样本，随机抽样，概率越高，抽到的概率就越高
samples = torch.multinomial(prob_dist, 1, replacement=True)

print("概率分布:", prob_dist)
print("抽取的样本索引:", samples)

# 显示每个样本对应的概率
print("每个样本对应的概率:", prob_dist[samples])

In [ ]:
def generate_text(model, start_string, max_len=1000, temperature=1.0, stream=True):
    input_eval = torch.Tensor([char2idx[char] for char in start_string]).to(dtype=torch.int64, device=device).reshape(1, -1) #bacth_size=1, seq_len长度是多少都可以 （1,5）
    hidden = None
    text_generated = [] #用来保存生成的文本
    model.eval()
    pbar = tqdm(range(max_len)) # 进度条
    print(start_string, end="")
    # no_grad是一个上下文管理器，用于指定在其中的代码块中不需要计算梯度。在这个区域内，不会记录梯度信息，用于在生成文本时不影响模型权重。
    with torch.no_grad():
        for i in pbar:#控制进度条
            logits, hidden = model(input_eval, hidden=hidden)
            # 温度采样，较高的温度会增加预测结果的多样性，较低的温度则更加保守。
            #取-1的目的是只要最后，拼到原有的输入上
            logits = logits[0, -1, :] / temperature
            # using multinomial to sampling
            probs = F.softmax(logits, dim=-1) #算为概率分布
            idx = torch.multinomial(probs, 1).item() #从概率分布中抽取一个样本,取概率较大的那些
            input_eval = torch.Tensor([idx]).to(dtype=torch.int64, device=device).reshape(1, -1) #把idx转为tensor
            text_generated.append(idx)
            if stream:
                print(idx2char[idx], end="", flush=True)
    return "".join([idx2char[i] for i in text_generated])


# load checkpoints
model.load_state_dict(torch.load("checkpoints/text_generation/best.ckpt", map_location="cpu"))
start_string = "All: " #这里就是开头，什么都可以
res = generate_text(model, start_string, max_len=1000, temperature=0.5, stream=True)

In [ ]:
res